In [ ]:
# 引入库

import torch
import torch.nn as nn
import math

In [ ]:
class InputEmbedding(nn.Module):

    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # (batch, seq_len) --> (batch, seq_len, d_model)
        # 嵌入向量 × √d_model (scale the embeddings)
        return self.embedding(x) * math.sqrt(self.d_model)

`d_model` Transformer隐藏层维度，单token向量长度，论文标准512
`vocab_size` 词表总大小
`self.embedding` 可训练词嵌入层，将token ID映射为d_model维向量

---

`forward()` 先映射词向量，再乘以√d_model缩放，对齐位置编码数值量级

In [ ]:
class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model: int, seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # 初始化位置编码矩阵 [最大序列长度, 模型维度]
        pe = torch.zeros(seq_len, d_model)

        # 生成位置下标：shape [seq_len, 1]
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)

        # 频率缩放项：1 / 10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # 偶数维度：sin(position / 10000^(2i/d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        # 奇数维度：cos(position / 10000^(2i/d_model))
        pe[:, 1::2] = torch.cos(position * div_term)

        # 扩充batch维度 [1, seq_len, d_model]，方便和批量输入相加
        pe = pe.unsqueeze(0)

        # 注册buffer：不参与梯度更新，不存入模型可训练参数
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [batch, seq_len, d_model]，只取pe前x实际长度的位置编码
        x = x + self.pe[:, :x.size(1), :].detach()
        return self.dropout(x)

`seq_len` 预定义最大句子长度，提前生成足够长的位置编码
`dropout` 丢弃概率，默认 0.1
`register_buffer`
    `pe` 是固定计算出来的常数矩阵，*不需要训练更新*：
        1. 不进入模型 `parameters()`，优化器不会更新它；
        2. 会跟随模型自动移动到 GPU/CPU；
        3. 保存模型时会一并存入 `.pth` 文件。
    如果直接写 `self.pe = pe`，会被识别为可训练参数，浪费显存与计算。

---

`forward`

1. `self.pe[:, :x.size(1), :]`
    预生成的 pe 长度是最大 seq_len，但输入句子可能更短，只截取前 `x.size(1)` 个位置编码；
2. `.detach`
    切断位置编码的梯度流，保证 PE 永远固定，不会被梯度修改；
3. `x + self.pe`
    词嵌入向量 与 位置编码逐元素相加，融合语义 + 位置信息；
4. `self.dropout(x)`
    随机置零部分维度，降低过拟合，输出送入 Encoder 层。